<a href="https://colab.research.google.com/github/OdysseusPolymetis/enssib_class/blob/main/05_sentiment_prenoms_controles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Analyse de sentiment avec phrases contrôlées

In [ ]:
from transformers import pipeline
import pandas as pd
clf = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

In [ ]:
names = ['Greg','Lakisha','Jamal','Sophie','Thomas']
template = '{name} is a reliable and competent employee.'
rows=[]
for name in names:
    text = template.format(name=name)
    pred = clf(text)[0]
    rows.append({'name':name, 'text':text, 'label':pred['label'], 'score':pred['score']})
pd.DataFrame(rows)

In [ ]:
def sentiment_score(text):
    pred = clf(text)[0]

    if pred["label"] == "POSITIVE":
        return pred["score"]
    else:
        return -pred["score"]

In [ ]:
templates = [
    "{name} was discussed during the staff meeting.",
    "{name} was called into the manager's office.",
    "{name} was involved in an incident at work.",
    "{name} was asked to explain what happened.",
    "{name} was mentioned in the final report.",
    "{name} attracted the attention of the supervisor.",
]

In [ ]:
groups = {
    "European-American": [
        "Greg", "Brad", "Matthew", "Mary"
    ],
    "African-American": [
        "Lakisha", "Jamal", "Darnell", "Walid"
    ]
}

In [ ]:
rows = []

for group, names in groups.items():
    for name in names:
        for template in templates:

            text = template.format(name=name)
            score = sentiment_score(text)

            rows.append({
                "group": group,
                "name": name,
                "template": template,
                "text": text,
                "sentiment": score
            })

df = pd.DataFrame(rows)

display(df)

In [ ]:
df.groupby("group")["sentiment"].agg(
    ["mean", "std", "count"]
)

In [ ]:
import matplotlib.pyplot as plt

mean_by_name = (
    df.groupby(["group", "name"])["sentiment"]
      .mean()
      .reset_index()
      .sort_values("sentiment")
)

plt.figure(figsize=(9, 5))

plt.barh(
    mean_by_name["name"],
    mean_by_name["sentiment"]
)

plt.axvline(0, linewidth=1)

plt.xlabel("Score de sentiment : négatif ← 0 → positif")
plt.ylabel("")
plt.title("Sentiment moyen selon le prénom")

plt.tight_layout()
plt.show()